In [0]:
TABLE_ORDER_BRONZE = "customer_360.bronze.orders"
TABLE_SILVER_ORDER = "customer_360.silver.orders"
TABLE_QUARANTINE_ORDER = "customer_360.quarantine.orders"
TABLE_ORDER_VIEWS = "customer_360.bronze.order_views"
ORDER_METRICS_TABLE = "customer_360.raw.order_silver_metrics"
PATH_ORDER_CHECKPOINTLOCATION_SILVER = (
    "/Volumes/customer_360/raw/source_files/checkpoints/silver/orders"
)
TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER_ORDER} (
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    product_id STRING NOT NULL,
    order_status STRING,
    quantity INT,
    total_amount DECIMAL(12,2),
    order_date TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_QUARANTINE_ORDER} (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_status STRING,
    quantity INT,
    total_amount DECIMAL(12,2),
    order_date TIMESTAMP,
    updated_at TIMESTAMP,
    failure_reason STRING,
    quarantined_at TIMESTAMP
)
USING DELTA
""")


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_ORDER_VIEWS} (
    order_id STRING NOT NULL,
    updated_at TIMESTAMP,
    viewed_at TIMESTAMP
)
USING DELTA
""")


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ORDER_METRICS_TABLE} (
    metric_time TIMESTAMP,
    batch_id BIGINT,
    query_name STRING,
    total_records BIGINT,
    valid_records BIGINT,
    invalid_records BIGINT,
    duplicate_records BIGINT
)
USING DELTA
""")

In [0]:
order_df = (
    spark
    .readStream
    .format("delta")
    .table(TABLE_ORDER_BRONZE)
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime


def process_order_dataframe(batch_df, batch_id):

    # Standardization + Data Quality Checks
    df = (
        batch_df
        .withColumn("order_id", trim(col("order_id")))
        .withColumn("customer_id", trim(col("customer_id")))
        .withColumn("product_id", trim(col("product_id")))
        .withColumn(
            "failure_reason",

            when(
                col("order_id").isNull(),
                "order_id is null"
            )

            .when(
                col("customer_id").isNull(),
                "customer_id is null"
            )

            .when(
                col("product_id").isNull(),
                "product_id is null"
            )

            .when(
                col("order_status").isNull(),
                "order_status is null"
            )

            .when(
                ~col("order_status").isin(
                    "PLACED",
                    "CONFIRMED",
                    "SHIPPED",
                    "DELIVERED",
                    "CANCELLED",
                    "RETURNED"
                ),
                "invalid order_status"
            )

            .when(
                col("quantity").isNull(),
                "quantity is null"
            )

            .when(
                col("quantity") <= 0,
                "quantity must be greater than 0"
            )

            .when(
                col("total_amount").isNull(),
                "total_amount is null"
            )

            .when(
                col("total_amount") <= 0,
                "total_amount must be greater than 0"
            )

            .when(
                col("order_date").isNull(),
                "order_date is null"
            )

            .when(
                col("updated_at").isNull(),
                "updated_at is null"
            )

            .otherwise(None)
        )
    )

    # Filter Clean Data

    valid_data = (
        df
        .filter(col("failure_reason").isNull())
        .drop("failure_reason")
    )


    
    # Invalid Data → Quarantine
    

    quarantine_data = (
        df
        .filter(col("failure_reason").isNotNull())
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    quarantine_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_QUARANTINE_ORDER)


    # Within-Batch Duplicate Detection
    # Duplicate Key = order_id + updated_at

    window = (
        Window
        .partitionBy(
            ["order_id", "updated_at"]
        )
        .orderBy(
            col("updated_at").asc()
        )
    )

    valid_data = (
        valid_data
        .withColumn(
            "rn",
            row_number().over(window)
        )
    )

    unique_data = (
        valid_data
        .filter(col("rn") == 1)
        .drop("rn")
    )

    duplicate_data = (
        valid_data
        .filter(col("rn") > 1)
        .drop("rn")
    )


    
    # Load Previously Processed Orders
    

    first_occurance = (
        spark
        .read
        .format("delta")
        .table(TABLE_ORDER_VIEWS)
    )


    # Find Previously Seen Orders

    seen_data = (
        first_occurance
        .join(
            unique_data,
            on=["order_id", "updated_at"],
            how="inner"
        )
        .select([
            "order_id",
            "customer_id",
            "product_id",
            "order_status",
            "quantity",
            "total_amount",
            "order_date",
            "updated_at"
        ])
    )


    # Find New Orders

    silver_df = (
        unique_data
        .join(
            first_occurance,
            on=["order_id", "updated_at"],
            how="left_anti"
        )
        .select([
            "order_id",
            "customer_id",
            "product_id",
            "order_status",
            "quantity",
            "total_amount",
            "order_date",
            "updated_at"
        ])
    )


    # Write New Orders → Silver

    silver_df.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_SILVER_ORDER)

    valid_data_count = silver_df.count()


    # Duplicate Data → Quarantine

    quarantine_data = (
        duplicate_data
        .unionByName(seen_data)
        .withColumn(
            "failure_reason",
            lit("duplicate record")
        )
        .withColumn(
            "quarantined_at",
            current_timestamp()
        )
    )

    quarantine_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_QUARANTINE_ORDER)

    duplicate_records = quarantine_data.count()


    # Store Processed Orders

    viewed_data = (
        silver_df
        .withColumn(
            "viewed_at",
            current_timestamp()
        )
        .select([
            "order_id",
            "updated_at",
            "viewed_at"
        ])
    )

    viewed_data.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable(TABLE_ORDER_VIEWS)


    # Write Silver Batch Metrics

    metric = [
        Row(
            metric_time=datetime.now(),
            batch_id=batch_id,
            query_name="order_silver",
            total_records=batch_df.count(),
            valid_records=valid_data_count,
            invalid_records=batch_df.count() - valid_data_count,
            duplicate_records=duplicate_records
        )
    ]

    metric_df = spark.createDataFrame(metric)

    metric_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ORDER_METRICS_TABLE)

In [0]:
query = (
    order_df
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(process_order_dataframe)
    .option(
        "checkpointLocation",
        PATH_ORDER_CHECKPOINTLOCATION_SILVER
    )
    .start()
)

query.awaitTermination()

In [0]:
import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="order_silver",
            batch_id=int(progress["batchId"]),
            input_rows=int(
                source.get("numInputRows", 0)
            ),
            input_rows_per_second=float(
                source.get("inputRowsPerSecond", 0.0)
            ),
            processed_rows_per_second=float(
                source.get("processedRowsPerSecond", 0.0)
            ),
            processing_time_ms=int(
                progress
                .get("durationMs", {})
                .get("triggerExecution", 0)
            )
        )
    )

if metrics:

    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)

In [0]:
spark.sql("SELECT * FROM customer_360.silver.orders").show()

spark.sql("SELECT * FROM customer_360.bronze.order_views").show()

spark.sql("SELECT * FROM customer_360.quarantine.orders").show()

spark.sql("SELECT * FROM customer_360.raw.order_silver_metrics").show()

spark.sql("SELECT * FROM customer_360.raw.stream_metrics").show()